<a href="https://colab.research.google.com/github/studentfaqih/HackFest-2024/blob/main/Polip_ConvNext.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

dataset_path = '/content/drive/My Drive/Biomedics Final Projects/dataset'

print("Isi folder dataset:")
print(os.listdir(dataset_path))

In [ ]:
import os
import shutil
import random
from sklearn.model_selection import train_test_split

base_dir = '/content/split_dataset'
train_dir = os.path.join(base_dir, 'train')
val_dir = os.path.join(base_dir, 'val')
test_dir = os.path.join(base_dir, 'test')

if os.path.exists(base_dir):
    shutil.rmtree(base_dir)

for cls in os.listdir(dataset_path):
    os.makedirs(os.path.join(train_dir, cls), exist_ok=True)
    os.makedirs(os.path.join(val_dir, cls), exist_ok=True)
    os.makedirs(os.path.join(test_dir, cls), exist_ok=True)

    cls_path = os.path.join(dataset_path, cls)
    images = os.listdir(cls_path)
    random.seed(42)
    random.shuffle(images)

    train_split, temp_split = train_test_split(images, test_size=0.4, random_state=42)
    val_split, test_split = train_test_split(temp_split, test_size=0.5, random_state=42)

    for split_name, split_list in zip(['train', 'val', 'test'],
                                      [train_split, val_split, test_split]):
        for img in split_list:
            src = os.path.join(cls_path, img)
            dst = os.path.join(base_dir, split_name, cls, img)
            shutil.copy(src, dst)

print("Dataset berhasil di-split (60% train, 20% val, 20% test)")

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications.convnext import preprocess_input

image_size = (224, 224)
batch_size = 32

train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    train_dir,
    image_size=image_size,
    batch_size=batch_size
)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    val_dir,
    image_size=image_size,
    batch_size=batch_size
)

test_ds = tf.keras.preprocessing.image_dataset_from_directory(
    test_dir,
    image_size=image_size,
    batch_size=batch_size
)

class_names = train_ds.class_names
print("Class names:", class_names)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.map(lambda x, y: (preprocess_input(x), y)).prefetch(AUTOTUNE)
val_ds = val_ds.map(lambda x, y: (preprocess_input(x), y)).prefetch(AUTOTUNE)
test_ds = test_ds.map(lambda x, y: (preprocess_input(x), y)).prefetch(AUTOTUNE)

In [ ]:
from tensorflow.keras.applications import ConvNeXtSmall
from tensorflow.keras import layers, models, optimizers

# ConvNeXtSmall
base_model = ConvNeXtSmall(
    include_top=False,
    input_shape=(224, 224, 3),
    weights="imagenet"
)
base_model.trainable = False

# Head Classification
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.3),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(2, activation='softmax')  #
])

model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
checkpoint_cb = tf.keras.callbacks.ModelCheckpoint(
    "best_model.h5", save_best_only=True, monitor='val_accuracy', mode='max'
)

earlystop_cb = tf.keras.callbacks.EarlyStopping(
    patience=5, restore_best_weights=True
)

# Training
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    callbacks=[checkpoint_cb, earlystop_cb]
)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid")
plt.rcParams['axes.facecolor'] = '#f9f9f9'

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].plot(history.history['accuracy'], label='Training Accuracy', color='#1f77b4', linewidth=2)
axes[0].plot(history.history['val_accuracy'], label='Validation Accuracy', color='#ff7f0e', linewidth=2)
axes[0].set_title('Akurasi Model', fontsize=16, fontweight='bold')
axes[0].set_xlabel('Epochs', fontsize=14)
axes[0].set_ylabel('Akurasi', fontsize=14)
axes[0].legend(loc='lower right', fontsize=12)
axes[0].grid(True, linestyle='--', alpha=0.6)

axes[1].plot(history.history['loss'], label='Training Loss', color='#2ca02c', linewidth=2)
axes[1].plot(history.history['val_loss'], label='Validation Loss', color='#d62728', linewidth=2)
axes[1].set_title('Loss Model', fontsize=16, fontweight='bold')
axes[1].set_xlabel('Epochs', fontsize=14)
axes[1].set_ylabel('Loss', fontsize=14)
axes[1].legend(loc='upper right', fontsize=12)
axes[1].grid(True, linestyle='--', alpha=0.6)

plt.suptitle('Performa Model selama Training', fontsize=18, fontweight='bold', y=1.05)
plt.tight_layout()
plt.show()

**TESTING**

In [ ]:
model.save("best_model.keras")

In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

model = tf.keras.models.load_model("best_model.keras")

class_names = ['polyp', 'polypnormal']

# Dapatkan label asli dan prediksi dari test set
y_true = []
y_pred = []
images_list = []

for images, labels in test_ds:
    preds = model.predict(images)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(preds, axis=1))
    images_list.extend(images.numpy())

y_true = np.array(y_true)
y_pred = np.array(y_pred)
images_array = np.array(images_list)

test_loss, test_acc = model.evaluate(test_ds)
print(f"Test Loss: {test_loss:.4f}, Test Accuracy: {test_acc:.4f}")

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix')
plt.show()

print("Classification Report:")
print(classification_report(y_true, y_pred, target_names=class_names))

In [ ]:
indices = {
    "TP": np.where((y_true == 1) & (y_pred == 1))[0],
    "TN": np.where((y_true == 0) & (y_pred == 0))[0],
    "FP": np.where((y_true == 0) & (y_pred == 1))[0],
    "FN": np.where((y_true == 1) & (y_pred == 0))[0],
}

sampled_indices = []
for key in indices:
    sampled_indices.extend(indices[key][:2])

plt.figure(figsize=(16, 6))
for i, idx in enumerate(sampled_indices):
    plt.subplot(2, 4, i + 1)
    plt.imshow(images_array[idx].astype("uint8"))
    true_label = class_names[y_true[idx]]
    pred_label = class_names[y_pred[idx]]
    plt.title(f"T: {true_label}\nP: {pred_label}", fontsize=10)
    plt.axis('off')
plt.suptitle("Visualisasi TP, TN, FP, FN", fontsize=14)
plt.tight_layout()
plt.show()